# EDA 03 · Auditoría del dataset de modelado

Este cuaderno **no mira el mercado**: mira la tabla que el modelo va a recibir.

Es el complemento de `EDA_02_precio.ipynb`, que analiza el precio real, horario y sin lags
para entender el dominio. Aquí la pregunta es otra: *¿qué ve realmente el modelo, y hay
algo en esa tabla que vaya a estropear los resultados sin dar la cara?*

Por eso la fuente tiene que ser **exactamente** la salida de `construir_dataset_maestro`, y
ninguna otra. Si este cuaderno reconstruyera el dataset por su cuenta, dejaría de auditar
nada.

**Las cinco preguntas que contesta**

1. ¿Qué forma tiene el dataset y qué columnas trae?
2. ¿Cuánta cobertura queda en cada variable *después* de aplicar los lags?
3. ¿Se parece el tramo de entrenamiento al de test, o el modelo va a evaluarse sobre otro
   mundo?
4. ¿Hay alguna columna que se cuele con información del futuro?
5. ¿Sobrevive alguna columna constante, duplicada o vacía?

> **Estado (19-ago-2026):** `construir_dataset_maestro.py` está roto — lee `esios_load_inter`,
> tabla que dejó de existir el 18-ago. Willy lo está arreglando. El cuaderno está escrito
> contra la *interfaz*, no contra la implementación: se puede ejecutar entero y, si el módulo
> todavía falla, cada sección lo dice y sigue adelante en vez de reventar.

## 0 · Preparación

In [1]:
import sys, warnings, inspect
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

# Ruta al modulo de Willy. Ajusta si tu cuaderno no cuelga de notebooks/
RUTA_MODELOS = Path("../modelos").resolve()
sys.path.append(str(RUTA_MODELOS))
print("Buscando el modulo en:", RUTA_MODELOS)
print("Existe:", RUTA_MODELOS.exists())

Buscando el modulo en: /Users/magui/git/edev_models/modelos
Existe: True


## 1 · Cargar el dataset, sea cual sea su interfaz

No damos por hecho el nombre de la función. El módulo se importa, se inspecciona y se
prueban los puntos de entrada conocidos por orden. Si ninguno funciona, se dice por qué y
el resto del cuaderno queda en modo «pendiente» en lugar de fallar.

In [2]:
DATASET = None
ORIGEN = None
ERROR = None

try:
    import construir_dataset_maestro as cdm
    print("Modulo importado.\n")

    publicos = [n for n in dir(cdm) if not n.startswith("_")]
    funciones = [n for n in publicos if callable(getattr(cdm, n))]
    constantes = [n for n in publicos if not callable(getattr(cdm, n))]

    print("FUNCIONES DISPONIBLES")
    for n in funciones:
        try:
            print(f"  {n}{inspect.signature(getattr(cdm, n))}")
        except (TypeError, ValueError):
            print(f"  {n}(...)")

    print("\nCONSTANTES (las listas de columnas viven aqui)")
    for n in constantes:
        v = getattr(cdm, n)
        if isinstance(v, (list, tuple, dict, set)):
            print(f"  {n}  ({len(v)}) -> {list(v)[:6]}{' ...' if len(v) > 6 else ''}")
        elif isinstance(v, (str, int, float)):
            print(f"  {n} = {v!r}")

    # Puntos de entrada conocidos, en orden de preferencia.
    for nombre in ("construir_dataset_diario", "construir_dataset",
                   "construir_espina_horaria", "main"):
        f = getattr(cdm, nombre, None)
        if f is None or not callable(f):
            continue
        try:
            DATASET = f()
            ORIGEN = nombre
            print(f"\n-> dataset construido con {nombre}()")
            break
        except Exception as e:
            print(f"\n   {nombre}() falla: {type(e).__name__}: {e}")

    if DATASET is None:
        ERROR = "Ninguna funcion de construccion devolvio un dataset."

except Exception as e:
    ERROR = f"{type(e).__name__}: {e}"
    print("No se pudo importar el modulo.")
    print(" ", ERROR)

if DATASET is None:
    print("\n" + "=" * 70)
    print("  DATASET NO DISPONIBLE — el cuaderno seguira, seccion a seccion,")
    print("  avisando de lo que no puede comprobar.")
    print("=" * 70)

Modulo importado.

FUNCIONES DISPONIBLES
  Path(*args, **kwargs)
  construir_dataset_diario(solo_filas_validas: bool = True, incluir_clima: bool = True, incluir_columnas_pendientes: bool = False) -> pandas.DataFrame
  construir_espina_horaria(start: str = '2020-01-01', end: str = '2026-08-15') -> pandas.DataFrame
  dividir_train_val_test(dataset: pandas.DataFrame)
  load_config()

CONSTANTES (las listas de columnas viven aqui)
  COLS_AUTOCONSUMO_PREV  (2) -> ['c_autoconsumo_prev', 'autoconsumo_estimado']
  COLS_CAPACIDAD_DISPONIBLE  (6) -> ['hydro_mw', 'pump_mw', 'nuclear_mw', 'coal_antracita_mw', 'ccgt_mw', 'fuel_mw']
  COLS_CLIMA  (9) -> ['t2m_mean', 'd2m_mean', 'wind10_mean', 'wind100_mean', 'wind_gust10_mean', 'ssrd_mean'] ...
  COLS_COMMODITIES  (3) -> ['gas_mibgas', 'co2_eua_dec', 'gas_ttf_m1']
  COLS_DEMANDA_REAL  (7) -> ['ree_load', 'entsoe_load', 'ree_netflow_fr', 'ree_netflow_pt', 'ree_netflow_ma', 'total_net_flow_mw'] ...
  COLS_ENTSOE_REAL  (7) -> ['wind_mw', 'pumping_cons_

In [3]:
def requiere_dataset(func):
    """Ejecuta la comprobacion solo si hay dataset. Si no, lo dice y sigue."""
    def envoltorio(*a, **kw):
        if DATASET is None:
            print(f"[pendiente] {func.__name__}: hace falta el dataset.")
            if ERROR:
                print(f"            motivo: {ERROR}")
            return None
        return func(*a, **kw)
    return envoltorio


BANDERAS = []   # se va llenando con lo que haya que arreglar

def bandera(gravedad, texto):
    """gravedad: 'BLOQUEA' | 'REVISAR' | 'NOTA'"""
    BANDERAS.append((gravedad, texto))
    print(f"  [{gravedad}] {texto}")

## 2 · Inventario

Qué es exactamente lo que devuelve: forma, granularidad, periodo y tipos. Suena trivial,
pero la mitad de los errores de un dataset se ven aquí: un índice duplicado, una
granularidad que no es la que se creía, o columnas que llegaron como texto.

In [4]:
@requiere_dataset
def inventario():
    df = DATASET
    print(f"Origen        : {ORIGEN}()")
    print(f"Forma         : {df.shape[0]:,} filas x {df.shape[1]} columnas".replace(",", "."))

    # Localizar el eje temporal, este en el indice o en una columna
    if isinstance(df.index, pd.DatetimeIndex):
        t = df.index.to_series()
        print("Eje temporal  : el indice")
    else:
        cand = [c for c in df.columns
                if pd.api.types.is_datetime64_any_dtype(df[c]) or c in ("datetime", "fecha", "ts")]
        if not cand:
            bandera("REVISAR", "No encuentro el eje temporal: ni indice ni columna de fecha.")
            return
        t = df[cand[0]]
        print(f"Eje temporal  : columna '{cand[0]}'")

    t = pd.to_datetime(t)
    print(f"Periodo       : {t.min()}  ->  {t.max()}")
    print(f"Zona horaria  : {t.dt.tz if hasattr(t.dt, 'tz') else 'naive'}")

    dif = t.sort_values().diff().dropna()
    if not dif.empty:
        print(f"Paso mas comun: {dif.mode().iloc[0]}  "
              f"({(dif == dif.mode().iloc[0]).mean()*100:.1f}% de los pasos)")
        if dif.nunique() > 1:
            print("Otros pasos   :")
            print(dif.value_counts().head(5).to_string())

    dup = t.duplicated().sum()
    if dup:
        bandera("BLOQUEA", f"{dup} marcas de tiempo duplicadas.")

    esperado = pd.date_range(t.min(), t.max(), freq=dif.mode().iloc[0]) if not dif.empty else []
    if len(esperado):
        faltan = len(esperado) - t.nunique()
        if faltan > 0:
            bandera("REVISAR", f"faltan {faltan:,} marcas de tiempo del rango completo."
                    .replace(",", "."))

    print("\nTIPOS")
    print(df.dtypes.value_counts().to_string())
    objeto = df.select_dtypes("object").columns.tolist()
    if objeto:
        bandera("REVISAR", f"columnas de texto donde se esperaba numero: {objeto[:8]}")

inventario()

Origen        : construir_dataset_diario()
Forma         : 2.410 filas x 264 columnas
  [REVISAR] No encuentro el eje temporal: ni indice ni columna de fecha.


## 3 · Cobertura después de los lags

Ésta es la sección que justifica el cuaderno. Una variable puede tener cobertura perfecta
en la base de datos y quedarse coja en el dataset, porque el lag la desplaza fuera del
rango o porque el cruce por fecha no encaja.

Se mira además **por tramo**: una columna que está completa en entrenamiento y vacía en
test no da error, da un modelo que no funciona.

In [5]:
# Cortes del split. Copiados de construir_dataset_maestro; si el modulo los expone
# como constantes, se leen de alli en lugar de fijarlos aqui.
FIN_TRAIN = pd.Timestamp("2024-12-31")
FIN_VAL   = pd.Timestamp("2025-12-31")

for nombre in ("FIN_TRAIN", "FIN_VAL", "CORTE_TRAIN", "CORTE_VAL"):
    if "cdm" in dir() and hasattr(cdm, nombre):
        print(f"  el modulo define {nombre} = {getattr(cdm, nombre)}")


def eje_temporal(df):
    if isinstance(df.index, pd.DatetimeIndex):
        s = df.index.to_series()
    else:
        cand = [c for c in df.columns
                if pd.api.types.is_datetime64_any_dtype(df[c]) or c in ("datetime", "fecha", "ts")]
        s = df[cand[0]] if cand else pd.Series(pd.NaT, index=df.index)
    s = pd.to_datetime(s)
    # Normalizamos a naive para poder comparar con los cortes sin sorpresas
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_convert("Europe/Madrid").dt.tz_localize(None)
    return s


@requiere_dataset
def cobertura_por_tramo():
    df = DATASET
    t = eje_temporal(df)
    tramo = pd.Series(np.select(
        [t <= FIN_TRAIN, t <= FIN_VAL], ["train", "val"], default="test"), index=df.index)

    print("FILAS POR TRAMO")
    print(tramo.value_counts().reindex(["train", "val", "test"]).to_string())

    num = df.select_dtypes("number")
    cob = pd.DataFrame({
        tr: num[tramo == tr].notna().mean() * 100
        for tr in ["train", "val", "test"] if (tramo == tr).any()
    }).round(1)
    cob["global"] = (num.notna().mean() * 100).round(1)

    print("\nCOBERTURA POR COLUMNA Y TRAMO (%)")
    print(cob.sort_values("global").to_string())

    # Lo que de verdad importa: columnas que cambian de disponibilidad entre tramos
    if {"train", "test"}.issubset(cob.columns):
        salto = (cob["train"] - cob["test"]).abs()
        rotas = salto[salto > 20].sort_values(ascending=False)
        print("\nCOLUMNAS QUE CAMBIAN DE COBERTURA ENTRE TRAIN Y TEST (> 20 puntos)")
        if rotas.empty:
            print("  ninguna")
        else:
            print(rotas.round(1).to_string())
            for c in rotas.index:
                bandera("BLOQUEA", f"'{c}': cobertura {cob.loc[c,'train']}% en train "
                                   f"y {cob.loc[c,'test']}% en test.")

    vacias = cob[cob["global"] == 0].index.tolist()
    if vacias:
        bandera("REVISAR", f"columnas totalmente vacias: {vacias}")
    return cob

COBERTURA = cobertura_por_tramo()

FILAS POR TRAMO
train       NaN
val         NaN
test     2410.0

COBERTURA POR COLUMNA Y TRAMO (%)
                                  test  global
ree_gbattery_mw_max_lag7d         24.8    24.8
ree_cbattery_mw_max_lag7d         24.8    24.8
ree_cbattery_mw_min_lag7d         24.8    24.8
ree_cbattery_mw_mean_lag7d        24.8    24.8
ree_gbattery_mw_mean_lag7d        24.8    24.8
ree_gbattery_mw_min_lag7d         24.8    24.8
ree_gbattery_mw_mean_lag1d        25.0    25.0
ree_gbattery_mw_min_lag1d         25.0    25.0
ree_gbattery_mw_max_lag1d         25.0    25.0
ree_cbattery_mw_mean_lag1d        25.0    25.0
ree_cbattery_mw_min_lag1d         25.0    25.0
ree_cbattery_mw_max_lag1d         25.0    25.0
ntc_pt_exp_prev_mw_mean           87.4    87.4
ntc_pt_imp_prev_mw_max            87.4    87.4
ntc_pt_imp_prev_mw_min            87.4    87.4
ntc_pt_imp_prev_mw_mean           87.4    87.4
ntc_fr_exp_prev_mw_mean           87.4    87.4
ntc_fr_exp_prev_mw_min            87.4    87.4
ntc_pt_e

## 4 · Deriva entre tramos

El modelo se entrena en un mundo y se evalúa en otro. Aquí se mide cuánto se parecen.

No es una preocupación teórica: en la decisión **D-03** comprobamos que Red Eléctrica
incorporó la estimación de autoconsumo en diciembre de 2025, lo que afecta al **0 % de
train y al 100 % de test**. Cualquier columna con ese comportamiento debería aparecer aquí
sin que nadie la busque.

Se usa el **PSI** (*population stability index*), que compara dos distribuciones por
deciles. La convención habitual: por debajo de 0,10 estable; entre 0,10 y 0,25 conviene
mirar; por encima de 0,25 son distribuciones distintas.

In [6]:
def psi(base, nueva, bins=10):
    """Population stability index entre dos muestras."""
    base, nueva = base.dropna(), nueva.dropna()
    if len(base) < 100 or len(nueva) < 100:
        return np.nan
    cortes = np.unique(np.quantile(base, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return np.nan
    cortes[0], cortes[-1] = -np.inf, np.inf
    b = np.histogram(base, cortes)[0] / len(base)
    n = np.histogram(nueva, cortes)[0] / len(nueva)
    b, n = np.clip(b, 1e-6, None), np.clip(n, 1e-6, None)
    return float(np.sum((n - b) * np.log(n / b)))


@requiere_dataset
def deriva():
    df = DATASET
    t = eje_temporal(df)
    tr = df[t <= FIN_TRAIN].select_dtypes("number")
    te = df[t > FIN_VAL].select_dtypes("number")
    if tr.empty or te.empty:
        print("  No hay ambos tramos para comparar.")
        return

    filas = []
    for c in tr.columns.intersection(te.columns):
        filas.append({
            "columna": c,
            "media_train": tr[c].mean(),
            "media_test": te[c].mean(),
            "psi": psi(tr[c], te[c]),
        })
    d = pd.DataFrame(filas).set_index("columna")
    d["cambio_%"] = ((d["media_test"] - d["media_train"]) /
                     d["media_train"].abs().replace(0, np.nan) * 100)
    d = d.sort_values("psi", ascending=False)

    print("DERIVA TRAIN -> TEST (ordenado por PSI)")
    print(d.round(2).to_string())

    for c, fila in d[d["psi"] > 0.25].iterrows():
        bandera("REVISAR", f"'{c}': PSI {fila['psi']:.2f}, la media pasa de "
                           f"{fila['media_train']:.1f} a {fila['media_test']:.1f}.")
    return d

DERIVA = deriva()

  No hay ambos tramos para comparar.


## 5 · Búsqueda de fugas

Una fuga no da error: da resultados demasiado buenos. Dos comprobaciones baratas que cazan
la mayoría.

**La correlación sospechosa.** Si una variable correlaciona con el objetivo casi
perfectamente, o correlaciona más con el instante actual que con su propio pasado, es que
lleva dentro información que no estará disponible en el momento de predecir.

**El desplazamiento revelador.** Para una variable con lag correcto, la correlación con el
objetivo debería ser mayor en su versión desplazada que sin desplazar. Si ocurre al revés,
el lag no se aplicó.

Ninguna de las dos demuestra una fuga por sí sola —una previsión meteorológica correlaciona
legítimamente con el presente— pero señalan dónde mirar.

In [7]:
# Nombre del objetivo. Se detecta, pero fijalo a mano si la deteccion falla.
CANDIDATOS_OBJETIVO = ["precio", "price", "target", "y", "spot", "es_esios", "precio_medio"]

@requiere_dataset
def buscar_fugas():
    df = DATASET.select_dtypes("number")

    objetivo = None
    for c in df.columns:
        if any(k in c.lower() for k in CANDIDATOS_OBJETIVO):
            objetivo = c
            break
    if objetivo is None:
        print("  No identifico la columna objetivo. Fijala a mano en OBJETIVO = '...'")
        return
    print(f"Objetivo detectado: '{objetivo}'\n")

    y = df[objetivo]
    filas = []
    for c in df.columns:
        if c == objetivo:
            continue
        s = df[c]
        if s.notna().sum() < 100:
            continue
        filas.append({
            "columna": c,
            "corr": s.corr(y),
            "corr_desplazada": s.shift(1).corr(y),
        })
    f = pd.DataFrame(filas).set_index("columna")
    f["gana_sin_desplazar"] = f["corr"].abs() > f["corr_desplazada"].abs()
    f = f.reindex(f["corr"].abs().sort_values(ascending=False).index)

    print("CORRELACION CON EL OBJETIVO")
    print(f.round(4).to_string())

    for c, fila in f[f["corr"].abs() > 0.95].iterrows():
        bandera("BLOQUEA", f"'{c}' correlaciona {fila['corr']:.3f} con el objetivo. "
                           "Comprobar que no es el objetivo disfrazado.")

    print("\nRECORDATORIO — el criterio del proyecto")
    print("  El mercado diario cierra a las 12:00 del dia anterior. Cualquier dato")
    print("  publicado despues de esa hora no puede entrar sin lag. Las unicas")
    print("  variables que entran sin retardo son las PREVISIONES.")
    if "cdm" in dir() and hasattr(cdm, "COLS_SEGURAS_FORECAST"):
        print(f"\n  El modulo declara seguras: {cdm.COLS_SEGURAS_FORECAST}")
        faltan = [c for c in cdm.COLS_SEGURAS_FORECAST if c not in DATASET.columns]
        if faltan:
            bandera("REVISAR", f"declaradas seguras pero ausentes del dataset: {faltan}")
    return f

FUGAS = buscar_fugas()

Objetivo detectado: 'price_h00'

CORRELACION CON EL OBJETIVO
                                   corr  corr_desplazada  gana_sin_desplazar
columna                                                                     
price_h01                        0.9934           0.9234                True
price_h02                        0.9870           0.9204                True
price_h03                        0.9797           0.9142                True
price_h05                        0.9749           0.9223                True
price_h04                        0.9747           0.9146                True
price_h06                        0.9704           0.9265                True
price_h07                        0.9493           0.9182                True
price_h23                        0.9455           0.9748               False
price_h22                        0.9414           0.9686               False
price_h21                        0.9224           0.9503               False
price_h08      

## 6 · Columnas que no aportan

Constantes, casi constantes y duplicadas exactas. En D-04 ya localizamos seis columnas
constantes en las tablas de capacidad —datos correctos, varianza cero— que no deben llegar
al modelo. Aquí se comprueba si alguna sobrevivió.

In [8]:
@requiere_dataset
def columnas_inutiles():
    num = DATASET.select_dtypes("number")

    print("CONSTANTES O CASI")
    hay = False
    for c in num.columns:
        s = num[c].dropna()
        if s.empty:
            continue
        if s.nunique() == 1:
            bandera("REVISAR", f"'{c}' es constante en {s.iloc[0]}.")
            hay = True
        elif s.value_counts(normalize=True).iloc[0] > 0.99:
            v = s.value_counts(normalize=True)
            bandera("NOTA", f"'{c}' vale {v.index[0]} el {v.iloc[0]*100:.1f}% del tiempo.")
            hay = True
    if not hay:
        print("  ninguna")

    print("\nPAREJAS CON CORRELACION > 0,99 (colinealidad)")
    corr = num.corr().abs()
    vistas = set()
    hay = False
    for a in corr.columns:
        for b in corr.columns:
            if a >= b or (a, b) in vistas:
                continue
            vistas.add((a, b))
            if corr.loc[a, b] > 0.99:
                print(f"  {a}  <->  {b}   r = {corr.loc[a, b]:.5f}")
                hay = True
    if not hay:
        print("  ninguna")

columnas_inutiles()

CONSTANTES O CASI
  [REVISAR] 'gen_solar_pv_prev_mw_min' es constante en 0.0.
  [NOTA] 'ree_gbattery_mw_min_lag1d' vale 0.0 el 99.3% del tiempo.
  [NOTA] 'ree_gbattery_mw_min_lag7d' vale 0.0 el 99.3% del tiempo.
  [REVISAR] 'ssrd_mean_min' es constante en 0.0.

PAREJAS CON CORRELACION > 0,99 (colinealidad)
  price_h00  <->  price_h01   r = 0.99344
  price_h01  <->  price_h02   r = 0.99591
  price_h01  <->  price_h03   r = 0.99065
  price_h02  <->  price_h03   r = 0.99628
  price_h02  <->  price_h04   r = 0.99227
  price_h03  <->  price_h04   r = 0.99748
  price_h03  <->  price_h05   r = 0.99365
  price_h04  <->  price_h05   r = 0.99611
  price_h10  <->  price_h11   r = 0.99006
  price_h11  <->  price_h12   r = 0.99558
  price_h12  <->  price_h13   r = 0.99636
  price_h12  <->  price_h14   r = 0.99095
  price_h13  <->  price_h14   r = 0.99641
  price_h14  <->  price_h15   r = 0.99459
  price_h15  <->  price_h16   r = 0.99081
  price_h22  <->  price_h23   r = 0.99141
  autoconsumo_estima

## 7 · Resumen: lo que hay que arreglar

Todo lo que las secciones anteriores han marcado, junto y ordenado por gravedad.

- **BLOQUEA** — no se puede modelar así.
- **REVISAR** — probablemente esté mal, hay que mirarlo.
- **NOTA** — conviene saberlo.

In [9]:
print("=" * 74)
print("  BANDERAS")
print("=" * 74)

if not BANDERAS:
    print("\n  Ninguna. Si el dataset se ha construido, esta limpio segun estas pruebas.")
    print("  Si no se ha construido, es que no se ha comprobado nada todavia.")
else:
    for nivel in ("BLOQUEA", "REVISAR", "NOTA"):
        del_nivel = [t for g, t in BANDERAS if g == nivel]
        if del_nivel:
            print(f"\n{nivel}  ({len(del_nivel)})")
            for t in del_nivel:
                print(f"  - {t}")

print("\n" + "=" * 74)
print("  PENDIENTES CONOCIDOS, INDEPENDIENTES DE LO ANTERIOR")
print("=" * 74)
print("""
  1. Zonas horarias. Unas tablas usan timestamptz y otras timestamp naive.
     Los cruces pueden desplazar 1-2 h sin avisar. Es problema de la capa
     silver: si esa capa fija UTC para todo, desaparece solo.

  2. Lag seguro: 48 o 72 h. Depende del crontab del servidor. Cambia todas
     las features de retardo a la vez.

  3. Autoconsumo (D-03). ree_load y ree_gsolar_mw cambian de definicion en
     dic-2025: 0% de train afectado, 100% de test. Deberia salir en la
     seccion 4 con PSI alto.

  4. Columnas constantes (D-04). Correctas en la base, inutiles como
     variables. Excluir del conjunto de features, no borrar de la base.
""")

  BANDERAS

BLOQUEA  (6)
  - 'price_h01' correlaciona 0.993 con el objetivo. Comprobar que no es el objetivo disfrazado.
  - 'price_h02' correlaciona 0.987 con el objetivo. Comprobar que no es el objetivo disfrazado.
  - 'price_h03' correlaciona 0.980 con el objetivo. Comprobar que no es el objetivo disfrazado.
  - 'price_h05' correlaciona 0.975 con el objetivo. Comprobar que no es el objetivo disfrazado.
  - 'price_h04' correlaciona 0.975 con el objetivo. Comprobar que no es el objetivo disfrazado.
  - 'price_h06' correlaciona 0.970 con el objetivo. Comprobar que no es el objetivo disfrazado.

REVISAR  (3)
  - No encuentro el eje temporal: ni indice ni columna de fecha.
  - 'gen_solar_pv_prev_mw_min' es constante en 0.0.
  - 'ssrd_mean_min' es constante en 0.0.

NOTA  (2)
  - 'ree_gbattery_mw_min_lag1d' vale 0.0 el 99.3% del tiempo.
  - 'ree_gbattery_mw_min_lag7d' vale 0.0 el 99.3% del tiempo.

  PENDIENTES CONOCIDOS, INDEPENDIENTES DE LO ANTERIOR

  1. Zonas horarias. Unas tablas usa